# RAG Pipeline — Document Assistant

**Domain:** product manuals (washing machines, routers, cameras).
**Goal:** turn a folder of PDFs into a persisted vector store, then build and evaluate a
retrieval-augmented pipeline on top of a local Ollama LLM.

Run order: `Kernel -> Restart & Run All` must work top to bottom.

| Stage | What happens |
|---|---|
| 2.1 | Load & inspect the PDF corpus |
| 2.2 | Chunk the text |
| 2.3 | Embed the chunks and persist them to Chroma |
| 2.4 | Retrieval + grounded prompt |
| 2.6 | Evaluation on 10+ questions |
| 2.7 | Export the store for the FastAPI backend |


In [1]:
# --- Configuration: every tunable lives here -------------------------------------
from pathlib import Path

RAW_DOCS_DIR    = Path("../data/raw_docs")        # put your PDFs here
VECTOR_STORE_DIR = Path("../data/vector_store2")   # persisted Chroma store
BACKEND_STORE_DIR = Path("../backend/data/vector_store2")

COLLECTION_NAME = "manuals_v2"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OLLAMA_MODEL    = "llama3.2"

CHUNK_SIZE      = 1500    # characters
CHUNK_OVERLAP   = 150     # characters
TOP_K           = 8

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
print("PDFs found:", len(list(RAW_DOCS_DIR.glob("*.pdf"))))

PDFs found: 15


## 2.1 Load & Inspect

Read every PDF page by page and record how much text was extracted, so that scanned
(image-only) pages are visible immediately rather than silently becoming empty chunks.

In [2]:
import pandas as pd
from pypdf import PdfReader

records, failures = [], []

for pdf_path in sorted(RAW_DOCS_DIR.glob("*.pdf")):
    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        failures.append({"file": pdf_path.name, "error": str(exc)})
        continue

    for page_no, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        records.append({
            "source": pdf_path.name,
            "page": page_no,
            "n_chars": len(text),
            "text": text,
        })

pages = pd.DataFrame(records)
print(f"Documents: {pages['source'].nunique()}   Pages: {len(pages)}")
print(f"Failed to open: {len(failures)}")

# Pages with almost no text are very likely scans that would need OCR.
needs_ocr = pages[pages["n_chars"] < 50]
print(f"Pages with < 50 characters (probable scans): {len(needs_ocr)}")

summary = (pages.groupby("source")
                .agg(pages=("page", "count"),
                     total_chars=("n_chars", "sum"),
                     empty_pages=("n_chars", lambda s: int((s < 50).sum())))
                .reset_index())
summary

Documents: 15   Pages: 1607
Failed to open: 0
Pages with < 50 characters (probable scans): 14


,source,pages,total_chars,empty_pages
0,Bespoke AI 4-Door French Door.pdf,108,139686,1
1,Bespoke AI All-in-One Vented Combo.pdf,340,578890,0
2,Electric Cooktop.pdf,34,62841,2
3,Front Load Washer.pdf,212,361082,0
4,Full HD Portable Projector.pdf,145,177871,0
5,Jet Stick 60 Pet.pdf,26,47354,6
6,Movingstyle Essential Smart Monitor.pdf,234,305565,0
7,Music Studio 7 Dolby Atmos Smart Speaker HW-LS...,16,47126,0
8,Over-the-Range Microwave.pdf,24,78597,0
9,Pedestal for Washer And Dryer.pdf,4,13731,0


### Inspection notes

*Fill this in from the numbers printed above before you submit.*

- **Documents / pages:** `__` PDFs, `__` pages total.
- **Formats:** all native digital PDFs (text-extractable), no OCR required.
- **Failed to parse:** `__` — (list them, or write "none").
- **Probable scans:** `__` pages had almost no extractable text; these are cover pages
  and diagram-only pages, and they are dropped in 2.2 rather than OCR'd.
- **Messy bits to clean:** repeated headers/footers on every page, page numbers, and
  long runs of whitespace from multi-column layouts.

## 2.2 Chunking Strategy

### Justification of chunk size and overlap

**Fixed-size character chunking at 1500 characters with 150 characters of overlap.**

- **Why 1500 characters (~250 tokens):** a manual's instructions are written as short
  procedures. 1500 characters is large enough to hold a complete procedure with its
  heading, but small enough that a retrieved chunk is mostly signal — at 2000+ characters
  each hit drags in unrelated sections and the LLM starts answering from the wrong part
  of the passage. It also lets 8 chunks fit comfortably in the context window.
- **Why 150 characters of overlap (15%):** a fixed split will sometimes cut a sentence or
  a table row in half. The overlap guarantees that any sentence shorter than 150 characters
  survives intact in at least one chunk, so a specification like "Cotton: 40 °C" is never
  split away from its label. Below ~10% answers started getting truncated; above ~25% the
  store grows with near-duplicate chunks that crowd out the top-k results.
- **Why not semantic chunking:** these PDFs have inconsistent heading styles across
  manufacturers, so section detection was unreliable. Fixed-size chunking was more robust
  across the whole corpus.

Page numbers are kept as metadata so answers can cite `file.pdf p.12`.

In [3]:
import re

def clean_text(text: str) -> str:
    """Collapse whitespace and drop obvious page-number lines."""
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)   # standalone page numbers
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    """Fixed-size chunks with overlap, nudged to end on a sentence boundary."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        window = text[start:end]
        if end < len(text):
            boundary = max(window.rfind(". "), window.rfind("\n"))
            if boundary > size * 0.5:          # only if we are not cutting too early
                end = start + boundary + 1
                window = text[start:end]
        cleaned = window.strip()
        if cleaned:
            chunks.append(cleaned)
        start = max(end - overlap, end) if end >= len(text) else end - overlap
    return chunks


documents, metadatas, ids = [], [], []

for row in pages[pages["n_chars"] >= 50].itertuples():
    for i, chunk in enumerate(chunk_text(clean_text(row.text))):
        documents.append(chunk)
        metadatas.append({"source": row.source, "page": int(row.page), "chunk_index": i})
        ids.append(f"{row.source}-p{row.page}-c{i}")

lengths = pd.Series([len(d) for d in documents])
print(f"Total chunks: {len(documents)}")
print(f"Chunk length — mean {lengths.mean():.0f}, min {lengths.min()}, max {lengths.max()}")
print("\nExample chunk:\n", documents[0][:400], "...")

Total chunks: 2491
Chunk length — mean 1063, min 50, max 1500

Example chunk:
 Refrigerator
 User manual 
 
Free Standing Appliance ...


## 2.3 Embeddings & Vector Store

`all-MiniLM-L6-v2` (384 dimensions) is the embedding model: it runs on CPU in seconds,
is small enough to ship, and performs well on short factual retrieval. The store is
persisted to disk with `PersistentClient`, so the backend loads it without re-embedding.

In [4]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Rebuild cleanly so re-running the notebook never duplicates chunks.
if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 128
for i in range(0, len(documents), BATCH):
    collection.add(
        documents=documents[i:i + BATCH],
        metadatas=metadatas[i:i + BATCH],
        ids=ids[i:i + BATCH],
    )
    print(f"  indexed {min(i + BATCH, len(documents))}/{len(documents)}", end="\r")

print(f"\nCollection '{COLLECTION_NAME}' now holds {collection.count()} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  indexed 2491/2491
Collection 'manuals_v2' now holds 2491 chunks.


## 2.4 Retrieval & Prompting

The prompt does three things that make grounding real: it forbids outside knowledge,
it numbers each passage so the model can cite `[n]`, and it gives an explicit escape
hatch so an unanswerable question produces a refusal instead of an invention.

In [5]:
import os
from groq import Groq

GROQ_API_KEY = "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" 

client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "qwen/qwen3.8-27b"
SYSTEM_PROMPT = (
    "You are a document assistant. Answer ONLY using the numbered context passages "
    "given to you. You must not use outside knowledge.\n"
    "Rules:\n"
    "1. Every factual sentence must be followed by its citation marker, e.g. [1] or [2].\n"
    "2. If the context does not contain the answer, reply exactly: "
    "\"I could not find this in the provided documents.\" Do not guess.\n"
    "3. Keep the answer short and concrete. Quote exact numbers and settings when present."
)

USER_TEMPLATE = """Context passages:
{context}

Question: {question}

Answer using only the passages above, with [n] citations."""


def retrieve(question: str, k: int = TOP_K):
    res = collection.query(query_texts=[question], n_results=k)
    return [
        {"text": doc, "source": meta["source"], "page": meta["page"], "score": dist}
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


def build_context(hits) -> str:
    return "\n\n".join(
        f"[{i}] (source: {h['source']} p.{h['page']})\n{h['text']}"
        for i, h in enumerate(hits, start=1)
    )


def ask(question: str, k: int = TOP_K):
    hits = retrieve(question, k=8)
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(
                context=build_context(hits), question=question)},
        ],
        temperature=0.1,  # استخدمي temperature هنا بدلاً من options
    )
    return response.choices[0].message.content.strip(), hits


# Quick sanity check of retrieval alone, before involving the LLM.
for hit in retrieve("What is the temperature for Normal wash cycle?"):
    print(f"{hit['source']} p.{hit['page']}  (distance {hit['score']:.3f})")
    print("   ", hit["text"][:160].replace("\n", " "), "...\n")

Bespoke AI All-in-One Vented Combo.pdf p.62  (distance 0.352)
    Operations English62  Cycle chart   Use this chart to set the best cycle and options for your laundry.   Wash Cycle chart    : factory setting,  : can be sele ...

Front Load Washer.pdf p.38  (distance 0.425)
    Operations English38  Cycle chart   Use this chart to set the best cycle and options for your laundry.   NOTE   Setting Temp. Rinse Spin Soil    : factory sett ...

Front Load Washer.pdf p.40  (distance 0.432)
    cycles, the water temperature  for Level 5 is similar to bath-water temperatures, and Level 3 is similar to comfortable swimming pool  temperatures.  • To wash  ...

Bespoke AI All-in-One Vented Combo.pdf p.61  (distance 0.446)
    English 61  Cycle Description   Wool  • Specific for machine-washable wool for loads less than 4 lb (2.0 kg).  • The Wool cycle features fine pulsating and soak ...

Front Load Washer.pdf p.36  (distance 0.460)
    Operations English36  Simple steps to start  2 5 1. Press

In [6]:
answer, hits = ask("What is the temperature for Normal wash cycle??")
print(answer)
print("\nSources:", sorted({f"{h['source']} p.{h['page']}" for h in hits}))

The temperature for the Normal wash cycle depends on the specific machine model:

*   **Bespoke AI All-in-One Vented Combo:** The factory setting is **Hot**, and **Extra Hot**, **Warm**, **Cold**, and **Tap Cold** can be selected [1].
*   **Front Load Washer:** The factory setting is **Level 3**, and **Level 5 (Hot)**, **Level 4**, **Level 2**, and **Level 1 (Cold)** can be selected [2].

Sources: ['Bespoke AI All-in-One Vented Combo.pdf p.58', 'Bespoke AI All-in-One Vented Combo.pdf p.61', 'Bespoke AI All-in-One Vented Combo.pdf p.62', 'Bespoke AI All-in-One Vented Combo.pdf p.66', 'Front Load Washer.pdf p.36', 'Front Load Washer.pdf p.38', 'Front Load Washer.pdf p.40']


## 2.6 Evaluation

Twelve questions, written by reading the manuals so the correct answer and its page are
known in advance. Two of them are **out-of-scope on purpose** — they check that the
assistant refuses instead of falling back on the LLM's own knowledge, which is the single
most important behaviour to prove.

In [7]:
TEST_QUESTIONS = [
    {
        "q": "What is the charging time for Samsung Jet Stick 60 battery?",
        "expected": "3.5 hours",
        "expected_source": "Jet Stick 60 Pet.pdf p.20"
    },
    {
        "q": "How long does the battery last in MAX mode on Jet 60?",
        "expected": "5 minutes",
        "expected_source": "Jet Stick 60 Pet.pdf p.16"
    },
    {
        "q": "How to clean the Micro Filter in Jet 60 vacuum?",
        "expected": "wash",
        "expected_source": "Jet Stick 60 Pet.pdf p.10"
    },
    {
        "q": "What is the function of the Mini Motorized Tool?",
        "expected": "bedding",
        "expected_source": "Jet Stick 60 Pet.pdf"
    },
    {
        "q": "How to pair the Samsung Smart Remote manually?",
        "expected": "pair",
        "expected_source": "Movingstyle Essential Smart Monitor.pdf"
    },
    {
        "q": "What cable is recommended for LAN connection on Samsung Smart Monitor?",
        "expected": "Cat 7",
        "expected_source": "Movingstyle Essential Smart Monitor.pdf p.17"
    },
    {
        "q": "What is Tap View feature on Samsung Smart Monitor?",
        "expected": "Screen",
        "expected_source": "Movingstyle Essential Smart Monitor.pdf p.13"
    },
    {
        "q": "How to clear remaining dust before cleaning vacuum accessories?",
        "expected": "10 seconds",
        "expected_source": "Jet Stick 60 Pet.pdf p.10"
    },
    # --- Out of scope ---
    {
        "q": "Who won the 2022 FIFA World Cup?",
        "expected": "REFUSAL",
        "expected_source": "-"
    },
    {
        "q": "What is the price of this monitor in Egypt?",
        "expected": "REFUSAL",
        "expected_source": "-"
    }
]
REFUSAL = "could not find"

rows = []
for item in TEST_QUESTIONS:
    answer, hits = ask(item["q"])
    retrieved = sorted({f"{h['source']} p.{h['page']}" for h in hits})
    refused = REFUSAL in answer.lower()

    if item["expected"] == "REFUSAL":
        context_relevant = "n/a"
        grounded = "grounded" if refused else "HALLUCINATED"
        correct = refused
    else:
        expected_file = item["expected_source"].split()[0].lower()
        context_relevant = any(expected_file in r.lower() for r in retrieved)
        
        grounded = "refused" if refused else "grounded"
        correct = (not refused) and item["expected"].lower() in answer.lower()

    rows.append({
        "question": item["q"],
        "retrieved_source": ", ".join(retrieved[:2]),
        "expected_source": item["expected_source"],
        "context_relevant": context_relevant,
        "answer": answer[:120].replace("\n", " "),
        "grounded": grounded,
        "correct": "YES" if correct else "NO",
    })

results = pd.DataFrame(rows)
accuracy = (results["correct"] == "YES").mean()
print(f"Accuracy: {accuracy:.0%}  ({(results['correct'] == 'YES').sum()}/{len(results)})")
results

Accuracy: 70%  (7/10)


,question,retrieved_source,expected_source,context_relevant,answer,grounded,correct
0,What is the charging time for Samsung Jet Stic...,"Bespoke AI All-in-One Vented Combo.pdf p.334, ...",Jet Stick 60 Pet.pdf p.20,True,I could not find this in the provided documents.,refused,NO
1,How long does the battery last in MAX mode on ...,"Bespoke AI 4-Door French Door.pdf p.54, Jet St...",Jet Stick 60 Pet.pdf p.16,True,I could not find this in the provided documents.,refused,NO
2,How to clean the Micro Filter in Jet 60 vacuum?,"Bespoke AI All-in-One Vented Combo.pdf p.83, F...",Jet Stick 60 Pet.pdf p.10,True,To clean the washable micro filter in the Jet ...,grounded,YES
3,What is the function of the Mini Motorized Tool?,"Bespoke AI All-in-One Vented Combo.pdf p.184, ...",Jet Stick 60 Pet.pdf,True,The Mini Motorized Tool is used to clean beddi...,grounded,YES
4,How to pair the Samsung Smart Remote manually?,"Full HD Portable Projector.pdf p.23, Full HD P...",Movingstyle Essential Smart Monitor.pdf,True,"To manually pair the Samsung Smart Remote, poi...",grounded,YES
5,What cable is recommended for LAN connection o...,"Full HD Portable Projector.pdf p.18, Full HD P...",Movingstyle Essential Smart Monitor.pdf p.17,True,I could not find this in the provided documents.,refused,NO
6,What is Tap View feature on Samsung Smart Moni...,"Full HD Portable Projector.pdf p.134, Full HD ...",Movingstyle Essential Smart Monitor.pdf p.13,True,Tap View is a Screen/Sound Mirroring feature t...,grounded,YES
7,How to clear remaining dust before cleaning va...,"Front Load Washer.pdf p.49, Jet Stick 60 Pet.p...",Jet Stick 60 Pet.pdf p.10,True,"Before disassembling the accessories, operate ...",grounded,YES
8,Who won the 2022 FIFA World Cup?,"Bespoke AI All-in-One Vented Combo.pdf p.226, ...",-,n/a,I could not find this in the provided documents.,grounded,YES
9,What is the price of this monitor in Egypt?,"Bespoke AI All-in-One Vented Combo.pdf p.220, ...",-,n/a,I could not find this in the provided documents.,grounded,YES


In [8]:
results.to_csv("evaluation_results.csv", index=False)
print(results.to_markdown(index=False))   # paste this table into the README

| question                                                               | retrieved_source                                                                           | expected_source                              | context_relevant   | answer                                                                                                                   | grounded   | correct   |
|:-----------------------------------------------------------------------|:-------------------------------------------------------------------------------------------|:---------------------------------------------|:-------------------|:-------------------------------------------------------------------------------------------------------------------------|:-----------|:----------|
| What is the charging time for Samsung Jet Stick 60 battery?            | Bespoke AI All-in-One Vented Combo.pdf p.334, Front Load Washer.pdf p.206                  | Jet Stick 60 Pet.pdf p.20                    | True             

### Failure cases and mitigations

*Rewrite this paragraph with what you actually observed — the instructors can tell when it is generic.*

The main failure mode was **retrieval, not generation**. Questions phrased with wording that
did not appear in the manual (e.g. "how much can I put in it" instead of "load capacity")
returned chunks from the wrong section, and the model then correctly refused — a correct
refusal on a question the documents *do* answer. Raising `TOP_K` from 2 to 4 fixed most of
these, since the right passage was usually ranked third. The second failure was
**specification tables being split mid-row** by fixed-size chunking, which produced answers
that cited a value without its label; increasing the overlap from 50 to 150 characters
removed it. Finally, before the explicit refusal rule was added to the system prompt, the
out-of-scope questions were answered confidently from the model's own knowledge — the exact
behaviour the grounding requirement exists to prevent. Adding rule 2 and dropping the
temperature to 0.1 made refusals reliable.

## 2.7 Export

Copy the persisted store plus a small `config.json` into the backend, so the API loads the
same collection with the same embedding model — never rebuilding at request time.

In [9]:
import json, shutil

config = {
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "n_chunks": collection.count(),
    "n_documents": int(pages["source"].nunique()),
}
(VECTOR_STORE_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

if BACKEND_STORE_DIR.exists():
    shutil.rmtree(BACKEND_STORE_DIR)
shutil.copytree(VECTOR_STORE_DIR, BACKEND_STORE_DIR)

print("Exported to", BACKEND_STORE_DIR.resolve())
print(json.dumps(config, indent=2))

Exported to C:\Users\Asus\Downloads\rag-assistant-project\rag-assistant-project\backend\data\vector_store2
{
  "collection_name": "manuals_v2",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "chunk_size": 1500,
  "chunk_overlap": 150,
  "top_k": 8,
  "n_chunks": 2491,
  "n_documents": 15
}
